# Comparacion MLP vs regresion lineal

Carga los seis CSV de `data/mlp/`, los compara contra `lr_benchmark.csv` y selecciona el mejor MLP por combinacion de ventanas.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "util.py").exists():
    for parent in Path.cwd().parents:
        if (parent / "util.py").exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from util import load_benchmark


In [2]:
RESULTS_DIR = PROJECT_ROOT / "data" / "mlp"
EXPECTED_MODELS = [
    "mlp_1x64_relu",
    "mlp_1x100_relu",
    "mlp_2x100_relu",
    "mlp_3x200_relu",
    "mlp_2x100_dropout",
    "mlp_2x100_bn_l2",
]

paths = [RESULTS_DIR / f"{name}.csv" for name in EXPECTED_MODELS]
missing = [path for path in paths if not path.exists()]
if missing:
    print("Faltan CSV por generar:")
    for path in missing:
        print(" -", path)

available_paths = [path for path in paths if path.exists()]
all_results = pd.concat([pd.read_csv(path) for path in available_paths], ignore_index=True)
all_results


,model_name,input_window,output_window,MAE_train,MAE_val,MAE_test,epochs,n_params
0,mlp_1x64_relu,5,1,0.010535,0.009040,0.014994,500,8919
1,mlp_1x64_relu,5,5,0.004929,0.004167,0.006338,500,8919
2,mlp_1x64_relu,5,30,0.002120,0.001764,0.002689,500,8919
3,mlp_1x64_relu,5,90,0.001313,0.001026,0.001616,500,8919
4,mlp_1x64_relu,10,1,0.010080,0.009035,0.015601,500,16279
...,...,...,...,...,...,...,...,...
91,mlp_2x100_bn_l2,30,90,0.002706,0.002352,0.002770,200,82323
92,mlp_2x100_bn_l2,90,1,0.012537,0.009286,0.013385,200,220323
93,mlp_2x100_bn_l2,90,5,0.008090,0.004780,0.008194,200,220323
94,mlp_2x100_bn_l2,90,30,0.005067,0.002404,0.005117,200,220323


## Comparacion contra benchmark lineal

In [3]:
lr = load_benchmark("lr_benchmark")
lr_comp = lr.rename(columns={"MAE_test": "MAE_test_lr", "MAE_train": "MAE_train_lr"})

comparison = all_results.merge(
    lr_comp[["input_window", "output_window", "MAE_test_lr"]],
    on=["input_window", "output_window"],
    how="left",
)
comparison["delta_vs_lr"] = comparison["MAE_test"] - comparison["MAE_test_lr"]
comparison["pct_delta_vs_lr"] = 100 * comparison["delta_vs_lr"] / comparison["MAE_test_lr"]
comparison.sort_values(["input_window", "output_window", "MAE_test"])


,model_name,input_window,output_window,MAE_train,MAE_val,MAE_test,epochs,n_params,MAE_test_lr,delta_vs_lr,pct_delta_vs_lr
64,mlp_2x100_dropout,5,1,0.010278,0.009028,0.012710,200,24023,0.012384,0.000326,2.631836
48,mlp_3x200_relu,5,1,0.009168,0.009034,0.014438,200,108223,0.012384,0.002054,16.584940
16,mlp_1x100_relu,5,1,0.010344,0.009039,0.014628,500,13923,0.012384,0.002244,18.119279
0,mlp_1x64_relu,5,1,0.010535,0.009040,0.014994,500,8919,0.012384,0.002611,21.081611
32,mlp_2x100_relu,5,1,0.009897,0.009036,0.015833,500,24023,0.012384,0.003449,27.853043
...,...,...,...,...,...,...,...,...,...,...,...
47,mlp_2x100_relu,90,90,0.001278,0.001033,0.001425,500,219523,0.001518,-0.000093,-6.158931
63,mlp_3x200_relu,90,90,0.001271,0.001005,0.001461,200,499223,0.001518,-0.000057,-3.748029
15,mlp_1x64_relu,90,90,0.001332,0.001038,0.001483,500,134039,0.001518,-0.000035,-2.281255
31,mlp_1x100_relu,90,90,0.001298,0.001037,0.001611,500,209423,0.001518,0.000093,6.107647


## Mejor MLP por combinacion de ventanas

In [4]:
idx = comparison.groupby(["input_window", "output_window"])["MAE_test"].idxmin()
best_mlp = comparison.loc[idx].sort_values(["input_window", "output_window"]).reset_index(drop=True)
best_mlp


,model_name,input_window,output_window,MAE_train,MAE_val,MAE_test,epochs,n_params,MAE_test_lr,delta_vs_lr,pct_delta_vs_lr
0,mlp_2x100_dropout,5,1,0.010278,0.009028,0.012710,200,24023,0.012384,0.000326,2.631836
1,mlp_2x100_dropout,5,5,0.004771,0.004166,0.005774,200,24023,0.005625,0.000150,2.659874
2,mlp_2x100_dropout,5,30,0.002081,0.001764,0.002417,200,24023,0.002340,0.000077,3.273956
3,mlp_2x100_dropout,5,90,0.001277,0.001023,0.001414,200,24023,0.001271,0.000143,11.220080
4,mlp_2x100_dropout,10,1,0.009759,0.009017,0.012971,200,35523,0.012554,0.000417,3.320112
5,mlp_2x100_dropout,10,5,0.004466,0.004161,0.005852,200,35523,0.005698,0.000155,2.713440
6,mlp_2x100_dropout,10,30,0.001976,0.001758,0.002488,200,35523,0.002358,0.000130,5.493194
7,mlp_2x100_dropout,10,90,0.001245,0.001027,0.001412,200,35523,0.001282,0.000130,10.120314
8,mlp_2x100_dropout,30,1,0.009357,0.009019,0.012797,200,81523,0.012924,-0.000127,-0.985009
9,mlp_2x100_dropout,30,5,0.004379,0.004162,0.005754,200,81523,0.005877,-0.000123,-2.095626


In [5]:
best_mlp_mae = best_mlp.pivot(index="input_window", columns="output_window", values="MAE_test")
best_mlp_name = best_mlp.pivot(index="input_window", columns="output_window", values="model_name")
best_delta = best_mlp.pivot(index="input_window", columns="output_window", values="delta_vs_lr")

print("Mejor MAE test MLP")
display(best_mlp_mae)
print("Modelo ganador")
display(best_mlp_name)
print("Delta vs regresion lineal")
display(best_delta)


Mejor MAE test MLP


output_window,1,5,30,90
input_window,,,,
5,0.012710,0.005774,0.002417,0.001414
10,0.012971,0.005852,0.002488,0.001412
30,0.012797,0.005754,0.002463,0.001373
90,0.012829,0.005806,0.002427,0.001387


Modelo ganador


output_window,1,5,30,90
input_window,,,,
5,mlp_2x100_dropout,mlp_2x100_dropout,mlp_2x100_dropout,mlp_2x100_dropout
10,mlp_2x100_dropout,mlp_2x100_dropout,mlp_2x100_dropout,mlp_2x100_dropout
30,mlp_2x100_dropout,mlp_2x100_dropout,mlp_2x100_dropout,mlp_3x200_relu
90,mlp_2x100_dropout,mlp_2x100_dropout,mlp_2x100_dropout,mlp_2x100_dropout


Delta vs regresion lineal


output_window,1,5,30,90
input_window,,,,
5,0.000326,0.000150,0.000077,0.000143
10,0.000417,0.000155,0.000130,0.000130
30,-0.000127,-0.000123,0.000026,0.000021
90,-0.001267,-0.000543,-0.000202,-0.000131


## Heatmaps de mejores MLP

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].imshow(best_mlp_mae.values, cmap="viridis", aspect="auto")
axes[0].set_title("Mejor MAE test MLP")
axes[0].set_xlabel("Ventana salida")
axes[0].set_ylabel("Ventana entrada")
axes[0].set_xticks(range(len(best_mlp_mae.columns)))
axes[0].set_xticklabels(best_mlp_mae.columns)
axes[0].set_yticks(range(len(best_mlp_mae.index)))
axes[0].set_yticklabels(best_mlp_mae.index)
for i in range(best_mlp_mae.shape[0]):
    for j in range(best_mlp_mae.shape[1]):
        axes[0].text(j, i, f"{best_mlp_mae.values[i, j]:.4f}", ha="center", va="center", color="white", fontsize=9)
plt.colorbar(im, ax=axes[0])

limit = np.abs(best_delta.values).max()
im = axes[1].imshow(best_delta.values, cmap="RdBu_r", aspect="auto", vmin=-limit, vmax=limit)
axes[1].set_title("Delta mejor MLP - LR")
axes[1].set_xlabel("Ventana salida")
axes[1].set_ylabel("Ventana entrada")
axes[1].set_xticks(range(len(best_delta.columns)))
axes[1].set_xticklabels(best_delta.columns)
axes[1].set_yticks(range(len(best_delta.index)))
axes[1].set_yticklabels(best_delta.index)
for i in range(best_delta.shape[0]):
    for j in range(best_delta.shape[1]):
        axes[1].text(j, i, f"{best_delta.values[i, j]:+.4f}", ha="center", va="center", color="black", fontsize=9)
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()


/var/folders/py/c5_xfbqn469g5_844mv32gt40000gn/T/ipykernel_73226/1208366811.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Ranking global de arquitecturas

In [7]:
ranking = comparison.groupby("model_name").agg(
    mean_MAE_test=("MAE_test", "mean"),
    median_MAE_test=("MAE_test", "median"),
    wins=("model_name", lambda s: (best_mlp["model_name"] == s.name).sum()),
    mean_delta_vs_lr=("delta_vs_lr", "mean"),
).sort_values("mean_MAE_test")
ranking


,mean_MAE_test,median_MAE_test,wins,mean_delta_vs_lr
model_name,,,,
mlp_2x100_dropout,0.005618,0.004121,0,-0.000050
mlp_3x200_relu,0.006165,0.004479,0,0.000496
mlp_2x100_relu,0.006678,0.004705,0,0.001009
mlp_1x64_relu,0.006862,0.004597,0,0.001194
mlp_1x100_relu,0.006970,0.004950,0,0.001302
mlp_2x100_bn_l2,0.019015,0.008115,0,0.013346


## Mejor modelo entre LR y MLP

In [8]:
lr_as_model = lr.copy()
lr_as_model["model_name"] = "regresion_lineal"
lr_as_model["MAE_val"] = np.nan
lr_as_model["epochs"] = np.nan
lr_as_model["n_params"] = np.nan

common_cols = ["model_name", "input_window", "output_window", "MAE_train", "MAE_val", "MAE_test", "epochs", "n_params"]
all_with_lr = pd.concat([all_results[common_cols], lr_as_model[common_cols]], ignore_index=True)
idx = all_with_lr.groupby(["input_window", "output_window"])["MAE_test"].idxmin()
best_overall = all_with_lr.loc[idx].sort_values(["input_window", "output_window"]).reset_index(drop=True)
best_overall


,model_name,input_window,output_window,MAE_train,MAE_val,MAE_test,epochs,n_params
0,regresion_lineal,5,1,0.011821,NaN,0.012384,NaN,NaN
1,regresion_lineal,5,5,0.005440,NaN,0.005625,NaN,NaN
2,regresion_lineal,5,30,0.002192,NaN,0.002340,NaN,NaN
3,regresion_lineal,5,90,0.001263,NaN,0.001271,NaN,NaN
4,regresion_lineal,10,1,0.011796,NaN,0.012554,NaN,NaN
5,regresion_lineal,10,5,0.005418,NaN,0.005698,NaN,NaN
6,regresion_lineal,10,30,0.002181,NaN,0.002358,NaN,NaN
7,regresion_lineal,10,90,0.001258,NaN,0.001282,NaN,NaN
8,mlp_2x100_dropout,30,1,0.009357,0.009019,0.012797,200.0,81523.0
9,mlp_2x100_dropout,30,5,0.004379,0.004162,0.005754,200.0,81523.0
